In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.schema import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

import os
import json


c:\Users\Rakesh Kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.3.0)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
c:\Users\Rakesh Kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [7]:
# STEP 1: Load medicine JSON
with open("C:/Users/Rakesh Kumar/VSCode/Capstone/datasets/rag_json_docs/allopathy_drugs.json", "r", encoding="utf-8") as f:
    medicines = json.load(f)


# STEP 2: Convert medicine JSON → semantic text
def medicine_to_text(m):
    return f"""
MEDICATION: {m.get('entity', '')}

CLASS: {m.get('drug_class', '')}

DESCRIPTION:
{m.get('description', '')}

USES:
{", ".join(m.get('uses', []))}

MECHANISM OF ACTION:
{m.get('mechanism', '')}

DOSAGE:
Adults: {m.get('dosage', {}).get('adults', '')}
Children: {m.get('dosage', {}).get('children', '')}

ONSET:
{m.get('onset', '')}

DURATION:
{m.get('duration', '')}

FORMS:
{", ".join(m.get('forms', []))}

SIDE EFFECTS:
{", ".join(m.get('side_effects', []))}

WARNINGS:
{", ".join(m.get('warnings', []))}

INTERACTIONS:
{", ".join(m.get('interactions', []))}

COMBINATION NOTE:
{m.get('combination_note', '')}
""".strip()


# STEP 3: Create LangChain Documents
documents = []

for med in medicines:
    if not isinstance(med, dict):
        continue  # skip bad rows

    doc = Document(
        page_content=medicine_to_text(med),
        metadata={
            "medication": med.get("entity"),
            "class": med.get("drug_class"),
            "uses": med.get("uses", []),
            "aliases": med.get("ocr_aliases", []),
            "tags": med.get("uses", []) + [med.get("drug_class", "")]
        }
    )

    documents.append(doc)

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

vector_db = FAISS.from_documents(documents, embeddings)

retriever = vector_db.as_retriever()

vector_db.save_local("vector_db/allopathy_drugs_db")

C:\Users\Rakesh Kumar\AppData\Local\Temp\ipykernel_13548\3333191641.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [4]:
system_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the provided context. But do not mention it out loud to the user that u have answered based on the provided context.
    And being a medical assistant, do not tell what is the primary issue is. Just tell how to recover from it
    Context:
    {context}

    Question:
    {input}
    """
)

In [5]:
os.environ["GOOGLE_API_KEY"] = "<your_gemini_key>"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [6]:
document_chain = create_stuff_documents_chain(llm, system_prompt)

rag_chain = create_retrieval_chain(retriever, document_chain)

In [8]:
response = rag_chain.invoke({"input": "I fell tired and exhausted with low blood levels. What should I do?"})

print(response["answer"])

To help support your energy levels, reduce fatigue, and maintain healthy blood components, consider incorporating foods rich in certain essential nutrients:

*   **For energy and fatigue reduction:**
    *   **Vitamin B1 (Thiamin):** This vitamin helps your body convert food into energy and supports nervous system function. You can find it in wholegrain breads, fortified breakfast cereals, peas, beans, nuts, bananas, oranges, pork, grains, and liver.
    *   **Potassium:** This mineral supports muscle function and can help with fatigue. Good sources include bananas, broccoli, parsnips, Brussels sprouts, beans, pulses, nuts, seeds, fish, beef, chicken, and turkey.

*   **For supporting healthy blood components and energy:**
    *   **Vitamin B12 (Cobalamin):** This vitamin is crucial for red blood cell formation and helps your body release energy from food. Include foods like meat, fish, milk, cheese, eggs, and fortified breakfast cereals.
    *   **Iron:** This essential mineral is vit